# Coerência: original (ρ²) vs correta (Eq. 7) — figuras 2×2
Mesmo estilo do `plot_lista_simulacoes_open_system_2x2.ipynb` (métrica + g(t) + ⟨σz⟩ + ⟨n⟩),
mas o painel da métrica mostra a coerência **original** (tracejado) e a **correta** (sólido).

- Coerência lida de `results/coherence_old_vs_correct/` (stacks `var_aberto.npy` e `var_aberto_correct.npy`).
- Observáveis (g_t, ⟨σz⟩, ⟨n⟩) lidos do run original `results/lista_simulacoes_open_system/`.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize":(12,8),"axes.grid":True,"grid.alpha":0.25,"font.size":11})

NEW  = "results/coherence_old_vs_correct"                 # coerencia old + correct
ORIG_BASE = "results/lista_simulacoes_open_system"        # observaveis (g_t, sz, n)

SAVE_FIGURES = True
FIG_DIR = os.path.join(NEW, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
DPI = 200

In [ ]:
def latest_orig(base=ORIG_BASE):
    fs = sorted(glob.glob(os.path.join(base, "open_system_sims*")))
    if not fs:
        raise FileNotFoundError("Rode antes: python run_lista_simulacoes_open_system.py")
    return fs[-1]

ORIG = latest_orig()
ORIG

In [ ]:
def load_new(case):
    d = os.path.join(NEW, case)
    return {
        "t":  np.load(os.path.join(d,"t.npy")),
        "scan": np.load(os.path.join(d,"scan_values.npy")),
        "var_old": np.load(os.path.join(d,"var_aberto.npy")),
        "var_cor": np.load(os.path.join(d,"var_aberto_correct.npy")),
        "const_old": np.load(os.path.join(d,"const_aberto.npy")),
        "const_cor": np.load(os.path.join(d,"const_aberto_correct.npy")),
    }

def orig_labels(case):
    # rotulos originais (para achar os *_observables.csv com g_t, sz, n)
    return pd.read_csv(os.path.join(ORIG, case, "args_per_scan.csv"))

def load_obs(case, label):
    return pd.read_csv(os.path.join(ORIG, case, f"{label}_observables.csv"))

def coherence_cases():
    return sorted(os.path.basename(d) for d in glob.glob(os.path.join(NEW,"*"))
                  if os.path.isdir(d) and "coherence" in os.path.basename(d))
coherence_cases()

In [ ]:
def selected_indices(case, n):
    # replica o espirito do notebook 2x2: specific -> todos; gauss -> av; cos -> min e av
    if "specific" in case:
        return list(range(n))
    if "cos" in case:
        return sorted(set([min(n//4, n-1), min(n//2, n-1)]))
    return [min(n//2, n-1)]   # gauss (varia zeta ou T): indice 'av'

In [ ]:
def plot_case_2x2(case, save=SAVE_FIGURES, show=True):
    d = load_new(case)
    t = d["t"]
    labels = orig_labels(case)
    idxs = selected_indices(case, len(d["scan"]))

    fig, ax = plt.subplots(2, 2, figsize=(12,8), constrained_layout=True)
    am, ag, az, an = ax[0,0], ax[0,1], ax[1,0], ax[1,1]

    # ---- curva constante (referencia) ----
    obs_c = load_obs(case, "const_aberto")
    tc = obs_c["time"].values
    am.plot(t, d["const_old"], ls="--", lw=1.8, color="0.45", label="constante — original")
    am.plot(t, d["const_cor"], ls="-",  lw=1.8, color="0.15", label="constante — correta")
    ag.plot(tc, obs_c["g_t"], lw=1.8, color="0.3", label="constante")
    az.plot(tc, obs_c["expect_Z_qubit"], lw=1.6, color="0.45")
    an.plot(tc, obs_c["expect_N"], lw=1.6, color="0.45")

    # ---- curvas variaveis selecionadas ----
    colors = plt.cm.viridis(np.linspace(0.15,0.85,len(idxs)))
    for c, i in zip(colors, idxs):
        lab = str(labels.iloc[i]["label"])
        val = float(labels.iloc[i]["scan_value"])
        obs = load_obs(case, lab)
        tt = obs["time"].values
        am.plot(t, d["var_old"][i], ls="--", lw=1.8, color=c, label=f"val={val:.3g} — original")
        am.plot(t, d["var_cor"][i], ls="-",  lw=1.8, color=c, label=f"val={val:.3g} — correta")
        ag.plot(tt, obs["g_t"], lw=1.8, color=c, label=f"val={val:.3g}")
        az.plot(tt, obs["expect_Z_qubit"], lw=1.6, color=c)
        an.plot(tt, obs["expect_N"], lw=1.6, color=c)

    am.set_yscale("log"); am.set_ylim(1e-4, 1.5)
    am.set_ylabel(r"$C_q(t)$"); am.set_title("Coerência: original (tracejado) vs correta (sólido)")
    am.legend(fontsize=8, ncol=1)
    ag.set_ylabel(r"$g(t)$"); ag.set_title("Acoplamento"); ag.legend(fontsize=8)
    az.set_ylabel(r"$\langle\sigma_z\rangle$"); az.set_title("Valor esperado (qubit)"); az.set_xlabel("t")
    an.set_ylabel(r"$\langle n\rangle$"); an.set_title("Valor esperado (campo)"); an.set_xlabel("t")
    fig.suptitle(case, fontsize=12)

    if save:
        fig.savefig(os.path.join(FIG_DIR, f"{case}_2x2.png"), dpi=DPI, bbox_inches="tight")
        fig.savefig(os.path.join(FIG_DIR, f"{case}_2x2.pdf"), bbox_inches="tight")
    if show: plt.show()
    else: plt.close(fig)

## Um caso (exemplo)

In [ ]:
plot_case_2x2("only_dephasing_coherence_gauss_zeta", save=False, show=True)

## Rodar e salvar TODOS os casos de coerência

In [ ]:
for case in coherence_cases():
    try:
        plot_case_2x2(case, save=True, show=False)
        print("[ok]", case)
    except Exception as e:
        print("[erro]", case, e)
print("\nFiguras salvas em:", FIG_DIR)